# Precompute WildSat GritLM Embeddings (Unified Cache)

This notebook reproduces the precompute flow from `datasets/utils.py`:
- load `train.csv` and `val.csv`
- encode all texts once with GritLM
- save one unified cache tensor
- write `embedding_index` into split CSVs for index-based lookup

It reuses helper functions from `datasets.utils` and keeps that module unchanged.

In [1]:
import sys
sys.path.append("../..")

In [2]:
from pathlib import Path
import json

import pandas as pd
import torch

from modeling.text import TextEncoder
from align_datasets.utils import _resolve_text_column, _encode_texts, _sanitize_model_id

In [6]:
# Update these paths/settings as needed.
data_path = Path("/home/libe2152/data/wildsat")
text_model_id = "GritLM/GritLM-7B"
batch_size = 128
output_dir = data_path / "gritlm_cache"

csv = data_path / "geolocated_text_distribution_habitat_only.csv"
if not csv.exists():    
    raise FileNotFoundError(f"Expected CSV at {csv}.")

print("Using:")
print("-", csv)
print("- output_dir:", output_dir)

Using:
- /home/libe2152/data/wildsat/geolocated_text_distribution_habitat_only.csv
- output_dir: /home/libe2152/data/wildsat/gritlm_cache


In [8]:
df = pd.read_csv(csv)

train_text_col = _resolve_text_column(df, csv)

all_texts = [str(v) for v in df[train_text_col].tolist()]

print(f"total texts: {len(all_texts):,}")

total texts: 2,513


In [9]:
encoder = TextEncoder(
    backend="gritlm",
    model_id=text_model_id,
    trainable=False,
    projection_head="none",
    target_dim=None,
)

embeddings = _encode_texts(
    encoder=encoder,
    texts=all_texts,
    desc="Encoding all texts",
    batch_size=batch_size,
)
print("Embedding tensor shape:", tuple(embeddings.shape))

/home/libe2152/miniconda3/envs/xai/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Created GritLM: torch.bfloat16 dtype, mean pool, embedding mode, bbcc attn
----------Using 3 data-parallel GPUs----------


Encoding all texts: 100%|██████████| 20/20 [19:10<00:00, 57.55s/batch]

Embedding tensor shape: (2513, 4096)


In [11]:
output_dir.mkdir(parents=True, exist_ok=True)
cache_path = output_dir / f"all_gritlm_embeddings.pt"

print(cache_path)

/home/libe2152/data/wildsat/gritlm_cache/all_gritlm_embeddings.pt


In [12]:
csv_with_embedding_index = data_path / "geolocated_text_distribution_habitat_only_with_embedding_index.csv"

In [14]:
metadata_path = output_dir / f"gritlm_embeddings_metadata.json"

In [15]:
torch.save(embeddings, cache_path)

n = len(df)
df["embedding_index"] = list(range(0, n))

df.to_csv(csv_with_embedding_index, index=False)

meta = {
    "cache_path": str(cache_path),
    "csv": str(csv_with_embedding_index),
    "rows_total": int(embeddings.shape[0]),
    "dim": int(embeddings.shape[1]),
    "model_id": text_model_id,
    "backend": "gritlm",
}
metadata_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")

print(f"Unified cache saved to: {cache_path}")
print(f"CSV with embedding_index: {csv_with_embedding_index}")
print(f"Metadata saved to: {metadata_path}")

Unified cache saved to: /home/libe2152/data/wildsat/gritlm_cache/all_gritlm_embeddings.pt
CSV with embedding_index: /home/libe2152/data/wildsat/geolocated_text_distribution_habitat_only_with_embedding_index.csv
Metadata saved to: /home/libe2152/data/wildsat/gritlm_cache/gritlm_embeddings_metadata.json


In [ ]:
from sklearn.model_selection import train_test_split

# Same split seed as before.
random_state = 42
test_size = 0.1

indexed_df = pd.read_csv(csv_with_embedding_index)

train_df, val_df = train_test_split(
    indexed_df,
    test_size=test_size,
    random_state=random_state,
    shuffle=True,
)

train_out = data_path / "train_with_embedding_index.csv"
val_out = data_path / "val_with_embedding_index.csv"

train_df.to_csv(train_out, index=False)
val_df.to_csv(val_out, index=False)

print(f"Train rows: {len(train_df):,} -> {train_out}")
print(f"Val rows: {len(val_df):,} -> {val_out}")
print(f"Seed used: {random_state}, test_size: {test_size}")